# 0. Inicialização Geral

In [ ]:
import pandas as pd
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv("/content/drive/MyDrive/CD/AcidentesPorOcorrencia_entrega3.csv")

print(f"Dataset carregado: {df.shape[0]} linhas, {df.shape[1]} colunas")

Mounted at /content/drive
Dataset carregado: 83817 linhas, 23 colunas


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 83817 entries, 0 to 83816
Data columns (total 23 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   dia_semana              83817 non-null  object 
 1   uf                      83817 non-null  object 
 2   br                      83817 non-null  int64  
 3   km                      83817 non-null  float64
 4   municipio               83817 non-null  object 
 5   causa_acidente          83817 non-null  object 
 6   tipo_acidente           83817 non-null  object 
 7   classificacao_acidente  83817 non-null  object 
 8   fase_dia                83817 non-null  object 
 9   sentido_via             83817 non-null  object 
 10  condicao_metereologica  83817 non-null  object 
 11  tipo_pista              83817 non-null  object 
 12  tracado_via             83817 non-null  object 
 13  uso_solo                83817 non-null  object 
 14  ignorados               83817 non-null

## Configurações de Encoding e Scalling
> Da entrega passada

In [ ]:
!pip -q install category_encoders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 2.9 MB/s eta 0:00:00


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler
from sklearn.neighbors import KNeighborsClassifier
from category_encoders import BinaryEncoder

In [ ]:
# === 1. Ordinal Encoder ===
class OrdinalFaseDiaEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.ordem = {'Pleno dia': 0, 'Amanhecer': 1, 'Anoitecer': 2, 'Plena Noite': 3}

    def fit(self, X, y=None):
        return self # Não precisa aprender nada do treino, a regra é estática

    def transform(self, X):
        X = X.copy()
        X['fase_dia'] = X['fase_dia'].map(self.ordem)
        return X

# === 2. Cyclic Encoder ===
class CyclicDiaSemanaEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.ordem = {'segunda-feira': 0, 'terça-feira': 1, 'quarta-feira': 2,
                      'quinta-feira': 3, 'sexta-feira': 4, 'sábado': 5, 'domingo': 6}

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X['dia_semana_num'] = X['dia_semana'].str.lower().map(self.ordem)
        X['dia_semana_sin'] = np.sin(2 * np.pi * X['dia_semana_num'] / 7)
        X['dia_semana_cos'] = np.cos(2 * np.pi * X['dia_semana_num'] / 7)
        return X.drop(columns=['dia_semana', 'dia_semana_num'])

# === 3. Frequency Encoder ===
class FrequencyEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cols):
        self.cols = cols
        self.freqs_ = {}

    def fit(self, X, y=None):
        # Aqui evitamos o Data Leakage: calculamos a frequência apenas no momento do .fit() (Treino)
        for col in self.cols:
            self.freqs_[col] = X[col].value_counts(normalize=True)
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.cols:
            # Se vier uma categoria nova no teste, preenche com 0
            X[col] = X[col].map(self.freqs_[col]).fillna(0)
        return X

In [ ]:
# Definindo as colunas
colunas_be = ['tracado_via', 'tipo_acidente', 'sentido_via', 'condicao_metereologica', 'uso_solo', 'tipo_pista']
colunas_freq = ['uf', 'causa_acidente']
colunas_escalar = ['km', 'veiculos', 'latitude', 'longitude', 'mes', 'hora', 'dia_semana_sin', 'dia_semana_cos']

# Criando o pré-processador para o Scaler (Utilizando o melhor avaliado - Robust Scaler)
scaler_step = ColumnTransformer(
    transformers=[
        ('robust', RobustScaler(), colunas_escalar)
    ],
    remainder='passthrough'
)

pipeline_knn = Pipeline(steps=[
    ('ordinal_fase_dia', OrdinalFaseDiaEncoder()),
    ('cyclic_dia_semana', CyclicDiaSemanaEncoder()),
    ('frequency_encoder', FrequencyEncoder(cols=colunas_freq)),
    ('binary_encoder', BinaryEncoder(cols=colunas_be)),
    ('scaler', scaler_step)
])

# Não perder o nome das colunas nos testes de MI e FeatureImportance
pipeline_knn.named_steps['scaler'].set_output(transform="pandas")

ColumnTransformer(remainder='passthrough',
                  transformers=[('robust', RobustScaler(),
                                 ['km', 'veiculos', 'latitude', 'longitude',
                                  'mes', 'hora', 'dia_semana_sin',
                                  'dia_semana_cos'])])

---

# 1. Técnicas de Seleção de Features

## 1.1 Análise de Variância e Cardinalidade

In [ ]:
print(df['ignorados'].value_counts(normalize=True) * 100)

ignorados
0     72.777599
1     19.726309
2      4.797356
3      1.889831
4      0.437859
5      0.170610
6      0.099025
7      0.028634
8      0.017896
9      0.017896
12     0.015510
11     0.007158
10     0.007158
16     0.002386
14     0.001193
17     0.001193
18     0.001193
13     0.001193
Name: proportion, dtype: float64


> Embora seja uma coluna com muitos zeros (72,7%), essa variável ainda possui uma "riqueza" considerável se comparada ao cenário de 95% (baixíssima variância). **É necessário outras avaliações para verificar a importância dessa feature.**

In [ ]:
colunas_remover = [
    'regional',
    'delegacia',
    'uop',
    'municipio',
    'br'
]

df = df.drop(columns=[c for c in colunas_remover if c in df.columns])

print(f"Colunas restantes: {df.shape[1]}")
print(f"\nColunas atuais:")
print(df.columns.tolist())

Colunas restantes: 18

Colunas atuais:
['dia_semana', 'uf', 'km', 'causa_acidente', 'tipo_acidente', 'classificacao_acidente', 'fase_dia', 'sentido_via', 'condicao_metereologica', 'tipo_pista', 'tracado_via', 'uso_solo', 'ignorados', 'veiculos', 'latitude', 'longitude', 'mes', 'hora']


## Resultados

Removidas 5 colunas:

- `regional`, `delegacia`, `uop` — administrativas/identificadores, sem valor preditivo
- `municipio` — alta cardinalidade sem ganho sobre `uf` + `km`
- `br` — redundante com `uf` + `km`

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 83817 entries, 0 to 83816
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   dia_semana              83817 non-null  object 
 1   uf                      83817 non-null  object 
 2   km                      83817 non-null  float64
 3   causa_acidente          83817 non-null  object 
 4   tipo_acidente           83817 non-null  object 
 5   classificacao_acidente  83817 non-null  object 
 6   fase_dia                83817 non-null  object 
 7   sentido_via             83817 non-null  object 
 8   condicao_metereologica  83817 non-null  object 
 9   tipo_pista              83817 non-null  object 
 10  tracado_via             83817 non-null  object 
 11  uso_solo                83817 non-null  object 
 12  ignorados               83817 non-null  int64  
 13  veiculos                83817 non-null  int64  
 14  latitude                83817 non-null

Considerando a Redução de Dimensionalidade, a lista de features do KNN fica:

- Geográficas/Temporais:

  - `uf`: Unidade Federativa
  - `latitude`, `longitude`: Coordenadas geográficas exatas
  - `mes`: Mês da ocorrência
  - `hora`: Horário exato
  - `dia_semana`: Dia da semana
  - `sentido_via`: Direção do fluxo onde ocorreu o acidente
  - `fase_dia`: Estado de luminosidade (Amanhecer, pleno dia, anoitecer, plena noite)
  - `km`: Ponto quilométrico da via

- Contextuais:

  - `causa_acidente`: Fator determinante do acidente
  - `tipo_acidente`: Natureza da ocorrência (Ex: colisão frontal, capotamento)
  - `classificacao_acidente`: Gravidade (Sem vítimas, com vítimas feridas, com vítimas fatais)
  - `condicao_metereologica`: Condições climáticas no momento
  - `tipo_pista`: Características da pista (Simples, dupla ou múltipla)
  - `tracado_via`: Geometria da via (Reta, curva, cruzamento, etc.)
  - `uso_solo`: Classificação da área (Urbano ou Rural)

- Quantitativas:

  - `veiculos`: Quantidade de veículos envolvidos
  - `ignorados`: Quantidade de pessoas cujos dados ou estado não foram identificados

---

### Preparação dos dados para os testes de Seleção de Features

In [ ]:
X = df.drop(columns=['classificacao_acidente'])
y = df['classificacao_acidente']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train_enc = pipeline_knn.fit_transform(X_train, y_train)
X_test_enc = pipeline_knn.transform(X_test)

N_FEATURES = 10

## 1.2 Informação Mútua

In [ ]:
from sklearn.feature_selection import mutual_info_classif

mi_scores = mutual_info_classif(X_train_enc, y_train, random_state=42)
mi_series = pd.Series(mi_scores, index=X_train_enc.columns).sort_values(ascending=False)

# Pegamos o nome das Top 10 (N) colunas
colunas_selecionadas_pela_mi = mi_series.head(N_FEATURES).index.tolist()

# Mostrar features para análise
df_mi = mi_series.head(N_FEATURES).to_frame(name='MI_Score')
print("TOP 10 FEATURES: MUTUAL INFORMATION")
print("Captura relações lineares e não lineares independentes")
print("="*50)
print(df_mi)

## Resultados

O método de Mutual Information avaliou a dependência isolada entre cada variável e o nosso alvo (target). Como o MI é capaz de detectar relações lineares e não lineares, este Top 10 revela quais características do acidente contêm mais "informação pura" sobre o desfecho, independentemente de outras variáveis.

-   `causa_acidente`:
    A variável `remainder__causa_acidente` é a campeã absoluta com um score de **0.0667**. Ela tem mais que o triplo de importância da segunda colocada. Isso faz total sentido no domínio de negócio: a infração ou motivo que gerou o acidente (ex: excesso de velocidade, embriaguez, falha mecânica) é o fator que mais dita a gravidade ou a natureza do evento final.

- `latitude`e `longitude`:
    `robust__latitude` (2º lugar) e `robust__longitude` (4º lugar) apareceram com alto destaque. Isso prova que **onde** o acidente ocorre é crucial. Trechos específicos de rodovias (curvas perigosas, serras, cruzamentos urbanos) concentram perfis específicos de acidentes.

-   `ignorados`:
    A presença da variável `remainder__ignorados` em 3º lugar (score 0.0193) mostra que a quantidade de informações ignoradas ou não preenchidas não é apenas um "ruído", mas sim um padrão. Pode indicar, por exemplo, que acidentes muito graves têm menos dados coletados na hora, ou que acidentes muito leves são subnotificados.

-  **Variáveis pós-encoding:**
    Categorias específicas de `tipo_acidente` (ramos 3, 2 e 4), além de `tipo_pista_1` e `uso_solo_1`, entraram no Top 10. Isso mostra que o *tipo* da colisão (ex: frontal vs. traseira) e a *geometria/localização* da via (reta, curva, área urbana/rural) possuem uma relação forte e direta com o desfecho, validando a engenharia de atributos realizada.

Este conjunto selecionado pelo MI aparenta ser altamente promissor para o KNN.

## 1.3 Feature Importance do Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_enc, y_train)

rf_importances = pd.Series(rf.feature_importances_, index=X_train_enc.columns).sort_values(ascending=False)

# Pegamos o nome das Top N colunas
colunas_selecionadas_pelo_rf = rf_importances.head(N_FEATURES).index.tolist()

# Mostrar features para análise
df_rf = rf_importances.head(N_FEATURES).to_frame(name='RF_Importance')
print("TOP FEATURES: RANDOM FOREST")
print("Captura interações complexas entre múltiplas variáveis")
print("="*50)
print(df_rf)

## Resltados

A Random Forest (RF) avalia a importância de uma variável com base no quanto ela ajuda a reduzir a "impureza" (geralmente Índice Gini) em todos os nós das centenas de árvores criadas. Isso significa que a RF é excelente para encontrar **combinações** de variáveis (ex: "Se for nesse Km E nessa hora = Acidente Grave").

### Principais Observações (RF):

-   `latitude`, `longitude`e `km`:
    O Top 3 é: `latitude`, `longitude` e `km`. Diferente do MI, a RF também trouxe fortemente as variáveis temporais: `hora`, `mes` e `dia_semana_sin`. O modelo nos diz que a combinação "Onde" e "Quando" cria os melhores caminhos lógicos para prever o alvo.

-   `causa_acidente`:
    Apesar de ter perdido o primeiro lugar absoluto que tinha no MI, a `causa_acidente` segue firmemente no Top 4, provando que é uma variável vital independentemente do algoritmo utilizado.

> A Random Forest tem um viés algorítmico conhecido: **ela tende a inflar a importância de variáveis contínuas com alta cardinalidade** (muitos valores únicos, como `km`, `latitude` e `hora`), pois essas variáveis oferecem infinitas possibilidades de "pontos de corte" (splits) para a árvore.

## 1.4 Verificando Desempenho

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

colunas_baseline = list(X_train_enc.columns)

experimentos = {
  "Baseline (Todas as features)": colunas_baseline,
  "Mutual Information (Top 10)": colunas_selecionadas_pela_mi,
  "Random Forest Importance (Top 10)": colunas_selecionadas_pelo_rf
}

modelo_knn = KNeighborsClassifier(n_neighbors=5, metric='euclidean', n_jobs=-1)

print("Iniciando testes...\n")

# Treina o KNN APENAS com as features de cada experimento e compara
for nome, features in experimentos.items():
  modelo_knn.fit(X_train_enc[features], y_train)
  y_pred = modelo_knn.predict(X_test_enc[features])

  # Relatório de Classificação (F1-score, Precision, Recall por classe)
  print("\n"+"="*50)
  print(f"\n=== RELATÓRIO DE CLASSIFICAÇÃO - {nome} ===")
  print(classification_report(y_test, y_pred))

  # Gerando a Matriz de Confusão
  print("\n=== GERANDO MATRIZ DE CONFUSÃO ===")
  cm = confusion_matrix(y_test, y_pred)

  disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=modelo_knn.classes_)

  fig, ax = plt.subplots(figsize=(8, 6))
  disp.plot(cmap='Blues', values_format='d', ax=ax)

  plt.title('Matriz de Confusão - Pipeline kNN com RobustScaler', pad=15)
  plt.xlabel('Previsão do Modelo')
  plt.ylabel('Valor Real')
  plt.grid(False)
  plt.show()

## Resultados

- **Baseline (Todas as features)**: O modelo sofreu com o que chamamos de "Maldição da Dimensionalidade". O KNN calcula a distância geométrica entre os pontos. Quando você tem muitas colunas (muitas features), essas distâncias ficam diluídas e perdem o sentido, dificultando a separação das classes menores ("Com Vítimas Fatais" e "Sem Vítimas").

- **Random Forest Importance (Top 10)** : Foi o pior cenário. A seleção baseada em árvores (Random Forest) avalia as variáveis de um jeito hierárquico que faz sentido para árvores de decisão, mas que muitas vezes não traduz bem para um algoritmo focado em distâncias espaciais como o KNN. O F1-score da classe "Com Vítimas Fatais" despencou de 0.20 para apenas 0.10.

- **Mutual Information (Top 10)**: Foi o melhor cenário. O MI avalia a dependência estatística direta entre cada feature e a variável alvo. Ele selecionou as colunas que realmente carregam sinal e eliminou o "ruído".



---

# 2. Redução de Dimensionalidade

## 2.1 Gráfico de Variância

In [ ]:
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# 1. Filtrando as 10 features selecionadas pelo MI (Melhor resultado)
X_mi = X_train_enc[colunas_selecionadas_pela_mi]

# Pré-PCA: Variância das Features
plt.figure(figsize=(10, 5))
variancias = X_mi.var().sort_values(ascending=False)
sns.barplot(x=variancias.values, y=variancias.index, palette='viridis')
plt.title("Variância de cada Feature (Antes do PCA)")
plt.xlabel("Variância")
plt.show()

## Resultados

- A Disparidade de Escala: A feature `robust__veiculos` domina completamente o gráfico com uma variância superior a 1.1, seguida por ignorados, longitude e latitude.

- A variável `remainder__causa_acidente`, que foi a campeã absoluta de importância no **Mutual Information**, tem uma variância quase nula (próxima a 0).

Nesse caso, rodar o PCA ou UMAP pode ser perigoso:

1.	O primeiro Componente Principal (PC1) vai ser praticamente um "espelho" da coluna `robust__veiculos`, só porque ela tem os maiores números soltos.

2.	O PCA vai ignorar a `causa_acidente` porque, matematicamente, ela não varia quase nada em termos de escala absoluta, mesmo ela sendo a variável que mais explica o target do dataset.

3.  Se a variável veiculos variae de 1 a 10 e a `causa_acidente` variar de 0.01 a 0.05, a diferença matemática na coluna veiculos vai dominar o cálculo da distância do UMAP.
> Isso faria com que o algoritmo agrupasse os acidentes baseando-se apenas na quantidade de veículos, ignorando as causas reais.

A solução é aplicar o StandardScaler antes do PCA e do UMAP.

## 2.2 Redução de Dimensionalidade (PCA e UMAP)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler_pca = StandardScaler()
X_mi_scaled = scaler_pca.fit_transform(X_mi)

In [ ]:
pca = PCA().fit(X_mi_scaled)
variancia_acumulada = np.cumsum(pca.explained_variance_ratio_)

# Exibindo quanto cada componente explica
for i, exp in enumerate(pca.explained_variance_ratio_):
    print(f"Componente {i+1}: {exp:.4f} ({variancia_acumulada[i]:.4f} acumulado)")

In [ ]:
import umap

redutor = umap.UMAP(n_components=2, random_state=42)
X_umap = redutor.fit_transform(X_mi_scaled)

## 2.3 Comparação

In [ ]:
sns.set_style("whitegrid")
sns.set_context("notebook", font_scale=1.1)
plt.figure(figsize=(12, 10))

sns.scatterplot(
    x=X_umap[:, 0],
    y=X_umap[:, 1],
    hue=y_train,
    palette='Set1',
    alpha=0.6,
    s=40,
    edgecolor='none'
)

plt.title("Visualização UMAP: Redução Não-Linear das 10 Features do MI\n(Para avaliação do potencial do KNN)", fontsize=16, fontweight='bold')
plt.xlabel("Dimensão 1 do UMAP", fontsize=12)
plt.ylabel("Dimensão 2 do UMAP", fontsize=12)

plt.legend(title='Classe Alvo', bbox_to_anchor=(1.05, 1), loc='upper left', frameon=True)

plt.tight_layout()
plt.show()

In [ ]:
# Pós-PCA: Variância das Features
plt.figure(figsize=(10, 5))
plt.plot(range(1, len(variancia_acumulada) + 1), variancia_acumulada, marker='o', linestyle='--')
plt.axhline(y=0.95, color='r', linestyle=':', label='95% de Variância')
plt.title("Variância Explicada Acumulada (Depois do PCA)")
plt.xlabel("Número de Componentes Principais")
plt.ylabel("Variância Explicada Total")
plt.legend()
plt.grid()
plt.show()

## Resultados

1. Análise do PCA (Gráfico de Variância e Dados)

As 10 features que o Mutual Information escolheu são altamente independentes entre si (ortogonais). Não há quase nenhuma redundância matemática entre elas.
-	Para atingir uma variância aceitável (acima de 90%), é preciso manter 8 dos 10 componentes.

**Conclusão**: O PCA não é útil para esse conjunto de variáveis. O ideal aqui é seguir com as 10 features originais padronizadas, sem redução linear.

---

2. Análise do UMAP (O Gráfico de Dispersão)

- Sobreposição Extrema: As classes estão completamente misturadas no espaço dimensional. Um acidente "Sem Vítimas" (azul) ou "Com Vítimas Fatais" (verde) está, em termos de distância (euclidiana/topológica), praticamente no mesmo lugar que um acidente "Com Vítimas Feridas" (vermelho).

-	Desbalanceamento Visível: A classe vermelha esmaga as outras visualmente.

**Conclusão**: Baseado apenas na "distância" usando essas 10 features, não há uma fronteira clara que separe os acidentes graves dos leves ou fatais.

## 2.4 Treinamento do Modelo (PCA)

In [ ]:
pca_teste = PCA(n_components=8, random_state=42)

X_test_mi = X_test_enc[colunas_selecionadas_pela_mi]
X_test_mi_scaled = scaler_pca.transform(X_test_mi)

X_train_pca = pca_teste.fit_transform(X_mi_scaled)
X_test_pca = pca_teste.transform(X_test_mi_scaled)

modelo_knn_pca = KNeighborsClassifier(n_neighbors=5, metric='euclidean', n_jobs=-1)
modelo_knn_pca.fit(X_train_pca, y_train)

# Fazendo as previsões
y_pred_pca = modelo_knn_pca.predict(X_test_pca)

# Gerando a Matriz de Confusão
print("\n"+"="*50)
print(f"\n=== RELATÓRIO DE CLASSIFICAÇÃO - PCA ===")
print(classification_report(y_test, y_pred_pca))

print("\n=== GERANDO MATRIZ DE CONFUSÃO ===")
cm_pca = confusion_matrix(y_test, y_pred_pca)

disp = ConfusionMatrixDisplay(confusion_matrix=cm_pca, display_labels=modelo_knn_pca.classes_)

fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(cmap='Blues', values_format='d', ax=ax)

plt.title('Matriz de Confusão - PCA (8 Components)', pad=15)
plt.xlabel('Previsão do Modelo')
plt.ylabel('Valor Real')
plt.grid(False)
plt.show()

## Resultados

Apesar de a Acurácia Geral (0.77) ser idêntica nos dois casos, o modelo com PCA é superior ao modelo MI puro.
  - Com Vítimas Fatais: O F1-Score subiu de 0.21 (MI) para 0.23 (PCA). O recall subiu de 0.16 para 0.18.
  -	Sem Vítimas: O F1-Score subiu de 0.33 (MI) para 0.36 (PCA). O recall subiu de 0.24 para 0.27.

1. A Ilusão da Acurácia: O modelo atingiu 77% de acurácia, o que parece ótimo. Mas ele conseguiu isso sendo "preguiçoso".

2. O Domínio da Classe Feridas: Como existem 12.999 casos de "Com Vítimas Feridas", o KNN viciou nessa classe. Ele acertou 93% (recall) de todos os acidentes com feridos simplesmente porque chutou essa classe na grande maioria das vezes.

3. A Tragédia das Minorias: Ao chutar "Feridas" para tudo, ele puniu severamente as classes minoritárias. O recall de "Fatais" é de apenas 0.18 no melhor cenário e de 0.27 para "Sem Vítimas".

In [ ]:
import umap
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# n_components para 5 (o suficiente para extrair relações não lineares, sem amassar demais os dados)
umap_modelo = umap.UMAP(n_components=5, random_state=42)

X_train_umap = umap_modelo.fit_transform(X_mi_scaled)
X_test_umap = umap_modelo.transform(X_test_mi_scaled)

modelo_knn_umap = KNeighborsClassifier(n_neighbors=5, metric='euclidean', n_jobs=-1)
modelo_knn_umap.fit(X_train_umap, y_train)

# Fazendo as previsões
y_pred_umap = modelo_knn_umap.predict(X_test_umap)

# Gerando a Matriz de Confusão
print("\n"+"="*50)
print(f"\n=== RELATÓRIO DE CLASSIFICAÇÃO - UMAP ===")
print(classification_report(y_test, y_pred_umap))

print("\n=== GERANDO MATRIZ DE CONFUSÃO ===")
cm_umap = confusion_matrix(y_test, y_pred_umap)

disp = ConfusionMatrixDisplay(confusion_matrix=cm_umap, display_labels=modelo_knn_umap.classes_)

fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(cmap='Blues', values_format='d', ax=ax)

plt.title('Matriz de Confusão - UMAP', pad=15)
plt.xlabel('Previsão do Modelo')
plt.ylabel('Valor Real')
plt.grid(False)
plt.show()

## Resultados

Apesar de a Acurácia Geral (0.77) tambêm ser idêntica nos dois casos, o modelo com UMAP obteve algumas melhorias.
  - Com Vítimas Fatais: O F1-Score subiu de 0.23 (PCA) para 0.25 (UMAP). O recall subiu de 0.18 para 0.19.
  -	Sem Vítimas: O F1-Score desceu de 0.36 (PCA) para 0.33 (UMAP). O recall subiu de 0.27 para 0.24.

> Em resumo, o modelo obteve um melhor resultado na feature mais crítica.

1. Alta Capacidade de Compressão: O UMAP conseguiu manter a acurácia geral em 77% usando apenas metade das features originais (5 componentes).

2. O Viés Persiste: Assim como vimos no gráfico 2D, a "força gravitacional" da classe majoritária (Com Vítimas Feridas) ainda domina o modelo. O KNN continuou com um recall altíssimo (93%) nessa classe, pois na dúvida (e no espaço do UMAP, há muita dúvida e sobreposição), ele chuta a classe mais frequente.

3. Avanço nas Fatalidades: O UMAP conseguiu isolar levemente melhor os acidentes fatais. A precisão subiu para 0.36 e o recall para 0.19, o que significa que, topologicamente, o UMAP encontrou sutis agrupamentos não lineares de acidentes fatais que as features originais sozinhas não deixavam tão claros.